# Convolutional Neural Network Review

A CNN is built by blocks.

## 1. Convolution (Conv)

Takes a small filter (kernel) a slides the whole images, in each position performs

`multiply + sum`

this produces a number, and by moving generates a full map (feature map)

<div>
    <img src='../images/CONV2D.png' width="600">
</div>

With an input image of size $(h, w)$ and a kernel of size $(k_h, k_w)$, the output dimensions are given by:

$$
o_h = h - k_h + 1 \\
o_w = w - k_w + 1
$$

We will derive this formula in the next section. For now, it can be understood intuitively from how the kernel slides over the input.

In [ ]:
import torch

x = torch.Tensor([
    [45, 12, 5, 17],
    [22, 10, 35, 6],
    [88, 26, 51, 19],
    [9, 77, 42, 3]
])

f = torch.Tensor([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
])

def conv(x, f):
    # Input shape
    h, w = x.shape
    # Kernel shape
    k_h, k_w = f.shape
    # Output
    o_h = h - k_h + 1
    o_w = w - k_w + 1
    output = torch.zeros(o_h, o_w)
    for i in range(o_h):
        for j in range(o_w):
            # Extraer parche
            patch = x[i:i+k_h, j:j+k_w]
            # Multiplicación elemento a elemento
            r = patch * f
            # Suma
            output[i, j] = torch.sum(r)
    return output

output = conv(x, f)
print(output)

tensor([[-45., 103.],
        [-96., 133.]])


In [76]:
m = torch.nn.Conv2d(
    in_channels=1,
    out_channels=1,
    kernel_size=3
)

f = torch.tensor([
    [0, -1, 0],
    [-1, 5, -1],
    [0, -1, 0]
], dtype=torch.float32)

with torch.no_grad():
    m.weight[0, 0] = f
    m.bias.zero_()

x = torch.tensor([
    [45, 12, 5, 17],
    [22, 10, 35, 6],
    [88, 26, 51, 19],
    [9, 77, 42, 3]
], dtype=torch.float32)

# 🔥 AQUÍ estaba el detalle
input = x.unsqueeze(0).unsqueeze(0)

print("Weight shape:", m.weight.shape)
print("Input shape:", input.shape)

output_py = m(input)

print("Output:")
print(output_py)

Weight shape: torch.Size([1, 1, 3, 3])
Input shape: torch.Size([1, 1, 4, 4])
Output:
tensor([[[[-45., 103.],
          [-96., 133.]]]], grad_fn=<ConvolutionBackward0>)


### 2.1 Stride

Stride is the step of the kernel over the window.

When $S = 1$, the number of positions is:

$$
N - K + 1
$$

This comes from the fact that the last valid starting position is:

$$
N - K
$$

For example, in a $4 \times 4$ input with a $3 \times 3$ kernel:

$$
4 - 3 = 1
$$

Since indexing starts at $0$, we add $1$ to count all valid positions:

$$
(N - K) + 1
$$

---

When stride is greater than $1$, the kernel moves in steps of size $S$:

$$
0, S, 2S, 3S, \dots
$$

We want the largest index $i$ such that the kernel still fits:

$$
i \cdot S \leq N - K
$$

Solving for $i$:

$$
i \leq \frac{N - K}{S}
$$

The number of valid positions is (with floor):

$$
\left\lfloor \frac{N - K}{S} \right\rfloor + 1
$$

---

Therefore, the general formula (without padding) is:

$$
\text{output} = \left\lfloor \frac{N - K}{S} \right\rfloor + 1
$$

In [97]:
import math

def conv_output_size(h, w, k_h, k_w, stride=1):
    o_h = math.floor((h - k_h) / stride) + 1
    o_w = math.floor((w - k_w) / stride) + 1
    return o_h, o_w

## 2. Activation function

In [86]:
relu = torch.nn.ReLU()
z = relu(output)
z

tensor([[  0., 103.],
        [  0., 133.]])

# 3. Pooling

Reduce the size of the feature map. Similar to convolution, it slides a window, in the next example lest assume a stride of 2.

<div>
    <img src='../images/POOL.png' width="600">
</div>

In [110]:
import torch

x = torch.tensor(
    [
        [4, 9, 2, 5],
        [5, 6, 2, 4],
        [2, 4, 5, 4],
        [5, 6, 8, 4]
    ], 
    dtype=torch.float32
)

# Pool size
ps = 2

h, w = x.shape

stride = 2
po_h, po_w = conv_output_size(h, w, ps, ps, stride=stride)

p_max = torch.zeros((po_h, po_w))
p_avg = torch.zeros((po_h, po_w))

for i in range(po_h):
    for j in range(po_w):
        ii = i * stride
        jj = j * stride
        e = x[ii:ii+ps, jj:jj+ps]
        p_max[i, j] = torch.max(e)
        p_avg[i, j] = torch.mean(e)

print("Max pooling")
print(p_max)
print("Average pooling")
print(p_avg)

Max pooling
tensor([[9., 5.],
        [6., 8.]])
Average pooling
tensor([[6.0000, 3.2500],
        [4.2500, 5.2500]])
